# 03 · Líneas base y modelos candidatos

* Misma preparación (`src/pipeline.py::construir`), misma validación (`CVTemporal`, 5 pliegues por fecha) y
  misma métrica técnica (**PR-AUC** = average precision) para todos.
* Cada corrida queda en MLflow (experimento `ecu911-demanda-alta`) con parámetros, datos (tabla, fecha de
  corte, hash), métricas de train/CV/tiempo, tags (fase, familia, commit) y el pipeline con firma y ejemplo.
* El test (hold-out) **no se toca** en este notebook.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import src  # noqa: F401  (en Windows carga LightGBM antes que pyarrow)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.datos import *
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

In [ ]:
import time, mlflow
from sklearn.dummy import DummyClassifier
from src.pipeline import construir, ReglaEstacional, candidatos, pr_auc
from src.experimento import configurar, registrar_corrida, evaluar_cv, commit_git, EXPERIMENTO
print("MLflow:", configurar(), "| experimento:", EXPERIMENTO, "| commit:", commit_git())

obt = cargar_obt(); MD = meta(obt)
datos = modelable(construir_dataset(obt))
train, test = dividir(datos)
X, y = train[FEATURES], train["y"]
cv = CVTemporal(train.fecha)
print(MD, f"| train={len(X):,} positivos={y.mean():.2%}")
resultados = {}

## 1. Líneas base

1. **Ingenua:** `DummyClassifier(strategy="prior")` → su PR-AUC es la prevalencia de cada pliegue.
2. **De negocio (PSet 1):** regla estacional — alerta si la media de las 4 últimas ocurrencias del mismo día
   de la semana supera el P90 histórico. Su puntaje (`ratio_regla`) permite calcular PR-AUC igual que un modelo.

In [ ]:
resultados["dummy_prior"] = registrar_corrida("dummy_prior", construir(DummyClassifier(strategy="prior")), X, y, cv,
                                              fase="baseline", familia="ingenua", meta_datos=MD)
resultados["regla_estacional"] = registrar_corrida("regla_estacional", ReglaEstacional(), X, y, cv,
                                                   fase="baseline", familia="negocio", meta_datos=MD)
for k in resultados:
    r = resultados[k]; print(f"{k:18s} CV PR-AUC = {r['cv_pr_auc_media']:.4f} ± {r['cv_pr_auc_desv']:.4f}")

## 2. Candidatos con hiperparámetros por defecto (5 familias tabulares)

In [ ]:
familias = {}
for nombre, (familia, modelo) in candidatos().items():
    t0 = time.time()
    resultados[nombre] = registrar_corrida(nombre, construir(modelo), X, y, cv, fase="candidato",
                                           familia=familia, meta_datos=MD)
    familias[nombre] = familia
    r = resultados[nombre]
    print(f"{nombre:24s} CV={r['cv_pr_auc_media']:.4f}±{r['cv_pr_auc_desv']:.4f} train={r['train_pr_auc']:.4f} "
          f"brecha={r['brecha_train_cv']:+.4f} fit={r['segundos_entrenamiento']:.1f}s ({time.time()-t0:.0f}s total)")

## 3. State of the art: Chronos-Bolt (modelo fundacional generativo de series de tiempo)

Zero-shot: no se entrena; para cada fila de validación recibe la serie del cantón **hasta t** (máx. 512 días)
y genera cuantiles de n(t+3). La probabilidad de pico es P(n(t+3) > umbral_objetivo) interpolada de esos
cuantiles. Se evalúa en **los mismos 5 pliegues de validación** con la misma métrica. No tiene "métrica de
entrenamiento" (no ve etiquetas), así que la brecha no aplica.

In [ ]:
from src.sota import chronos_scores, CHRONOS_ID, CONTEXTO
with mlflow.start_run(run_name="chronos_bolt_small") as run:
    folds, t0 = [], time.time()
    oof_chronos = pd.Series(np.nan, index=train.index)
    for i, (a, b) in enumerate(cv.split()):
        s = chronos_scores(obt, train.iloc[b], HORIZONTE)
        oof_chronos.iloc[b] = s
        folds.append(pr_auc(y.iloc[b], s)); print(f"pliegue {i}: PR-AUC={folds[-1]:.4f}  ({time.time()-t0:.0f}s)")
    seg = time.time() - t0
    mlflow.log_params({"modelo_id": CHRONOS_ID, "contexto_dias": CONTEXTO, "modo": "zero-shot", "semilla": SEMILLA,
                       "cv_esquema": "CVTemporal", "cv_n_splits": 5, **MD.como_params()})
    met = {"cv_pr_auc_media": float(np.mean(folds)), "cv_pr_auc_desv": float(np.std(folds)),
           "segundos_entrenamiento": 0.0, "cv_segundos": seg,
           "ms_inferencia_por_1000": 1000 * seg / oof_chronos.notna().sum(),
           **{f"cv_pr_auc_fold{i}": f for i, f in enumerate(folds)}}
    mlflow.log_metrics(met)
    mlflow.set_tags({"fase": "candidato", "familia": "state_of_the_art", "modelo": "chronos_bolt_small",
                     "git_commit": commit_git(), "nota": "zero-shot; sin metrica de train; sin pipeline sklearn"})
    resultados["chronos_bolt_small"] = {**met, "run_id": run.info.run_id, "_folds": folds}
    familias["chronos_bolt_small"] = "state_of_the_art"
print(f"Chronos-Bolt CV = {met['cv_pr_auc_media']:.4f} ± {met['cv_pr_auc_desv']:.4f}")

## 4. LLM mediante prompting (Qwen2.5-1.5B-Instruct, local)

El problema **sí** se puede plantear como texto: se describe la historia de 28 días del cantón, el P90 del día
de semana objetivo y si es feriado, y se pregunta "¿superará el P90 en 3 días? Sí/No". El puntaje es la
probabilidad del token "Sí" frente a "No" (una sola pasada, sin generar texto).
Por costo (~1 s por prompt en CPU) se evalúa en una **muestra aleatoria de 600 filas del último pliegue de
validación**, y en esas mismas filas se comparan la regla y LightGBM (entrenado sólo con el train de ese pliegue).

In [ ]:
from src.sota import llm_scores, LLM_ID, prompt_fila
from lightgbm import LGBMClassifier
a, b = list(cv.split())[-1]
muestra = train.iloc[b].sample(600, random_state=SEMILLA)
print("Ejemplo de prompt:\n", prompt_fila(muestra.iloc[0], obt[obt.cod_canton == muestra.iloc[0].cod_canton]
      .set_index("fecha").n_emergencias.loc[:muestra.iloc[0].fecha].tail(28).to_numpy(), HORIZONTE))
t0 = time.time(); s_llm = llm_scores(obt, muestra, HORIZONTE); seg_llm = time.time() - t0
lgbm_fold = construir(LGBMClassifier(random_state=SEMILLA, verbose=-1)).fit(X.iloc[a], y.iloc[a])
comp_llm = pd.Series({
    "LLM (Qwen2.5-1.5B)": pr_auc(muestra.y, s_llm),
    "regla_estacional": pr_auc(muestra.y, muestra.ratio_regla),
    "lightgbm (default)": pr_auc(muestra.y, lgbm_fold.predict_proba(muestra[FEATURES])[:, 1]),
    "chronos_bolt_small": pr_auc(muestra.y, oof_chronos.loc[muestra.index]),
    "prevalencia (azar)": muestra.y.mean()}, name="PR-AUC en la muestra de 600")
with mlflow.start_run(run_name="llm_qwen2.5_1.5b_prompting"):
    mlflow.log_params({"modelo_id": LLM_ID, "n_muestra": len(muestra), "pliegue": 4, "semilla": SEMILLA,
                       "modo": "zero-shot prompting, P(Sí)/(P(Sí)+P(No))", **MD.como_params()})
    mlflow.log_metrics({"muestra_pr_auc": comp_llm.iloc[0], "muestra_pr_auc_regla": comp_llm.iloc[1],
                        "muestra_pr_auc_lightgbm": comp_llm.iloc[2], "segundos_inferencia": seg_llm,
                        "ms_inferencia_por_1000": 1000 * seg_llm / len(muestra) * 1000 / 1000})
    mlflow.set_tags({"fase": "candidato", "familia": "llm", "modelo": "llm_qwen_prompting", "git_commit": commit_git()})
    mlflow.log_text(prompt_fila(muestra.iloc[0], np.zeros(28), HORIZONTE), "plantilla_prompt.txt")
display(comp_llm.to_frame())
print(f"Tiempo LLM: {seg_llm:.0f}s para {len(muestra)} filas → {seg_llm/len(muestra):.2f}s/fila "
      f"(todo el test serían ~{seg_llm/len(muestra)*len(test)/3600:.1f} h)")

## 5. Resultados: entrenamiento vs CV, brecha, learning rate y tiempo

In [ ]:
def lr_de(nombre):
    m = resultados[nombre].get("_modelo")
    if m is None: return np.nan
    est = m.named_steps["modelo"] if hasattr(m, "named_steps") else m
    p = est.get_params()
    for k in ("learning_rate_init", "learning_rate", "eta"):
        if k in p and isinstance(p[k], (int, float)): return p[k]
    return 0.3 if type(est).__name__ == "XGBClassifier" else np.nan

tabla = pd.DataFrame([{
    "modelo": k, "familia": ("baseline" if k in ("dummy_prior", "regla_estacional") else familias.get(k)),
    "train_pr_auc": r.get("train_pr_auc", np.nan), "cv_media": r["cv_pr_auc_media"], "cv_desv": r["cv_pr_auc_desv"],
    "brecha": r.get("brecha_train_cv", np.nan), "learning_rate": lr_de(k),
    "seg_entrenamiento": r["segundos_entrenamiento"], "ms_inferencia_1000": r.get("ms_inferencia_por_1000", np.nan),
    "run_id": r["run_id"]} for k, r in resultados.items()]).sort_values("cv_media", ascending=False)
regla = resultados["regla_estacional"]["cv_pr_auc_media"]; dummy = resultados["dummy_prior"]["cv_pr_auc_media"]
tabla["supera_dummy"] = tabla.cv_media > dummy
tabla["supera_regla"] = tabla.cv_media > regla
tabla["diagnostico"] = np.select([tabla.brecha > 0.3, tabla.brecha > 0.05], ["sobreajuste severo", "sobreajuste moderado"], "brecha baja")
tabla.loc[tabla.brecha.isna(), "diagnostico"] = "n/a (sin entrenamiento)"
tabla.to_csv(RAIZ / "docs" / "resultados_candidatos.csv", index=False)
tabla

In [ ]:
# Comparación pareada por pliegue (los pliegues difieren mucho en dificultad: misma partición para todos)
fold = pd.DataFrame({k: r["_folds"] for k, r in resultados.items()}, index=[f"pliegue {i}" for i in range(5)])
display(fold.T.round(4))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
t = tabla.set_index("modelo")
t.cv_media.plot.barh(xerr=t.cv_desv, ax=ax[0], color="#4C72B0"); ax[0].axvline(regla, c="r", ls="--", label="regla"); ax[0].axvline(dummy, c="grey", ls=":", label="dummy")
ax[0].set(title="PR-AUC en CV (media ± desv.)", xlabel="PR-AUC"); ax[0].legend(); ax[0].invert_yaxis()
tt = t.dropna(subset=["train_pr_auc"])
ax[1].scatter(tt.cv_media, tt.train_pr_auc)
for k, r in tt.iterrows(): ax[1].annotate(k, (r.cv_media, r.train_pr_auc), fontsize=8)
ax[1].plot([0.05, 1], [0.05, 1], "k--", lw=.8); ax[1].set(title="Train vs CV (diagonal = sin brecha)", xlabel="CV", ylabel="train", yscale="log")
fold.plot(marker="o", ax=ax[2], legend=False); ax[2].set(title="PR-AUC por pliegue", ylabel="PR-AUC")
ax[2].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIG / "candidatos_resultados.png", bbox_inches="tight"); plt.show()

In [ ]:
# Top 3 para tuning: modelos tabulares ajustables (se excluyen baselines, Chronos y el LLM, que no tienen
# hiperparámetros de entrenamiento comparables) ordenados por CV media.
ajustables = tabla[tabla.familia.isin(["lineal", "bagging", "boosting", "red_neuronal"])]
top3 = ajustables.head(3)[["modelo", "familia", "cv_media", "cv_desv", "run_id"]]
top3.to_csv(RAIZ / "docs" / "top3_para_tuning.csv", index=False)
top3